# Whissle Arc on Amazon SageMaker: sample notebook

This notebook deploys a **Whissle Arc** model package from AWS Marketplace and runs it two ways:

1. a **real-time endpoint**, invoked with a WAV file (and with JSON carrying a names list), and
2. a **batch transform** job over WAV files in S3,

then deletes everything it created. It works for both CPU products:

| `PRODUCT` | model package | returns |
|---|---|---|
| `"transcriber"` | Arc Transcriber (Arc Mini) | `verbose_json`: transcript, word times, linked names, number values, speakers, utterance tags, events |
| `"combined"` | Arc Transcriber + Tagger | the same plus `arc_tags`: PII spans and redaction, safety flags, intent, dialogue act, events |

The container makes no outbound calls and runs with network isolation. Your audio stays in your account.

**Costs.** The endpoint and the batch job bill per instance-hour (SageMaker infrastructure plus the listing's software
price) until the clean-up cell runs. Run the last section when you are done.

**Prerequisites**
- An AWS account with permission to create SageMaker models, endpoints and transform jobs, and an S3 bucket.
- A SageMaker execution role (inside SageMaker Studio / a notebook instance, `get_execution_role()` finds it).
- The SageMaker Python SDK v2 (`sagemaker>=2.200,<3`); the cell below installs it if needed.
- No API keys or secrets are needed. Do not paste credentials into this notebook.

## 1. Subscribe to the listing

1. Open the product's page in AWS Marketplace and choose **Continue to subscribe**.
2. Review the pricing and the End User License Agreement, then choose **Accept offer**.
3. Choose **Continue to configuration**, pick your Region and copy the **model package ARN** shown there.
4. Paste it into `MODEL_PACKAGE_ARNS` below (one per Region you use).

In [ ]:
%pip install --quiet "sagemaker>=2.200,<3" boto3

In [ ]:
import json, os, time, urllib.request
from pathlib import Path

import boto3
import sagemaker
from sagemaker import ModelPackage, get_execution_role

PRODUCT = "transcriber"          # or "combined"

# From the Marketplace configuration page (step 1). Placeholders: replace with the ARNs for your subscription.
MODEL_PACKAGE_ARNS = {
    "transcriber": {
        "us-east-1": "arn:aws:sagemaker:us-east-1:<seller-account>:model-package/arc-transcriber-mini-<id>",
    },
    "combined": {
        "us-east-1": "arn:aws:sagemaker:us-east-1:<seller-account>:model-package/arc-transcriber-tagger-<id>",
    },
}

session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()            # or your own bucket name
prefix = f"arc-sample/{PRODUCT}"

try:
    role = get_execution_role()
except ValueError:
    role = "arn:aws:iam::<your-account>:role/<sagemaker-execution-role>"   # outside SageMaker: set your role ARN

model_package_arn = MODEL_PACKAGE_ARNS[PRODUCT].get(region)
assert model_package_arn and "<" not in model_package_arn, f"set the model package ARN for {region} in MODEL_PACKAGE_ARNS"

REALTIME_INSTANCE = "ml.c6i.xlarge"           # listed real-time types: ml.c6i.large-4xlarge, ml.m6i.xlarge/2xlarge
TRANSFORM_INSTANCE = "ml.c6i.2xlarge"         # listed transform types: ml.c6i.xlarge-4xlarge, ml.m6i.2xlarge
stamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"arc-{PRODUCT}-{stamp}"
endpoint_name = f"arc-{PRODUCT}-{stamp}"
print(region, bucket, model_package_arn)

## 2. Sample audio

`example_synth.wav` is a short synthetic basketball commentary clip (16 kHz mono, about 1 MB). The cell uses a local
copy if you cloned the repository, otherwise downloads it from the public GitHub repository. Any WAV (PCM 8/16/24/32-bit
or float, any sample rate or channel count) or raw PCM16 works; MP3, FLAC, Ogg and MP4 must be converted first.

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/WhissleAI/arc_transcriber/main/examples/example_synth.wav"
local = Path("../../../examples/example_synth.wav")
wav_path = local if local.exists() else Path("example_synth.wav")
if not wav_path.exists():
    urllib.request.urlretrieve(SAMPLE_URL, wav_path)
audio = wav_path.read_bytes()
assert audio[:4] == b"RIFF" and audio[8:12] == b"WAVE", "not a WAV file"
print(wav_path, len(audio), "bytes")   # real-time payloads must stay under 6 MB (about 3 min of 16 kHz PCM16)

## 3. Create a model from the model package and deploy a real-time endpoint

Marketplace model packages always run with network isolation; it is set explicitly here as well. Deployment takes
several minutes.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=session,
    name=model_name,
    enable_network_isolation=True,
)
model.deploy(initial_instance_count=1, instance_type=REALTIME_INSTANCE, endpoint_name=endpoint_name)
print("endpoint", endpoint_name, "is InService")

## 4. Invoke the endpoint with a WAV

`Content-Type: audio/wav` returns `verbose_json`. Options travel in the custom attributes
(`domain=nba;response_format=text;sample_rate=8000`); `Accept: text/plain` returns plain text.

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

resp = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="audio/wav",
    Accept="application/json",
    CustomAttributes="domain=nba",
    Body=audio,
)
result = json.loads(resp["Body"].read())
print(result["text"])
print("duration:", result.get("duration"), "s  model:", result.get("model"))
for n in result.get("numbers", [])[:5]:
    print("number:", n.get("heard"), "->", n.get("kind"), n.get("value"), n.get("value2"))
for s in result.get("segments", [])[:3]:
    print("segment", s["id"], s.get("speaker"), (s.get("intent") or {}).get("label"), "|", s["text"])

### With a names list (JSON request)

Names link mentions to your spelling, including partial mentions and aliases.

In [ ]:
import base64

names = [
    {"name": "Marcus Delaney", "kind": "person", "group": "Harbor City Hawks"},
    {"name": "Theo Brandt", "kind": "person", "group": "Harbor City Hawks", "aliases": ["Brandt"]},
    {"name": "Owen Castellano", "kind": "person", "group": "Riverside Comets"},
    {"name": "Harbor City Hawks", "kind": "team", "aliases": ["Hawks"]},
    {"name": "Riverside Comets", "kind": "team", "aliases": ["Comets"]},
]
request = {"audio": base64.b64encode(audio).decode(), "domain": "nba", "names": names, "response_format": "verbose_json"}
if PRODUCT == "combined":
    request["tagger"] = {"domain": "talk", "redact": "mask"}

resp = runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json",
                               Accept="application/json", Body=json.dumps(request))
result = json.loads(resp["Body"].read())
for n in result.get("names", []):
    print(f'{n["heard"]!r:28} {n["state"]:8} -> {n.get("entity")}')

### Combined product: tags on the transcript, and text-only tagging

`arc_tags` holds Arc Tagger's output on the transcript's words. A request with `text` (no audio) goes straight to the
tagger. Skipped for `PRODUCT = "transcriber"`.

In [ ]:
if PRODUCT == "combined":
    tags = result.get("arc_tags", {})
    for sp in tags.get("spans", [])[:10]:
        print(f'{sp["type"]:32} {sp.get("score", 0):.2f}  {sp.get("text")}')
    for u in tags.get("utterances", [])[:3]:
        print("intent:", [i["tag"] for i in u.get("intent", []) if i.get("on")], "|", u.get("text"))

    text_req = {"text": "Hi, this is Maria Lopez. My number is 555 867 5309.", "domain": "talk", "redact": "mask"}
    resp = runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json",
                                   Accept="application/json", Body=json.dumps(text_req))
    print(json.loads(resp["Body"].read()).get("redacted_text"))
else:
    print("transcriber package: no arc_tags (use PRODUCT = 'combined')")

## 5. Batch transform

One WAV per S3 object gives one JSON per object. The same model runs the job on its own instances; the endpoint is
not used. For many short clips, JSON Lines of `{"audio": "<base64>"}` with `SplitType="Line"` and
`strategy="MultiRecord"` also works.

In [ ]:
input_uri = session.upload_data(str(wav_path), bucket=bucket, key_prefix=f"{prefix}/batch-in")
input_prefix = input_uri.rsplit("/", 1)[0] + "/"
output_uri = f"s3://{bucket}/{prefix}/batch-out/{stamp}/"

transformer = model.transformer(
    instance_count=1,
    instance_type=TRANSFORM_INSTANCE,
    strategy="SingleRecord",
    max_payload=100,                 # MB per record; clips up to 900 s by default
    max_concurrent_transforms=2,     # = ARC_WORKERS
    output_path=output_uri,
    accept="application/json",
)
transformer.transform(data=input_prefix, content_type="audio/wav", split_type="None", wait=True, logs=False)
print("job", transformer.latest_transform_job.name, "done ->", output_uri)

In [ ]:
s3 = boto3.client("s3", region_name=region)
out_prefix = output_uri.replace(f"s3://{bucket}/", "")
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=out_prefix).get("Contents", []):
    body = json.loads(s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read())
    print(obj["Key"], "->", body.get("text", "")[:200])

## 6. Clean up

Deletes the endpoint (and its config) and the model so nothing keeps billing. To stop being charged for the
subscription itself, cancel it in AWS Marketplace (*Manage subscriptions*). The S3 objects under `prefix` are left in
place; the last line removes them if you want.

In [ ]:
from sagemaker.predictor import Predictor

Predictor(endpoint_name, sagemaker_session=session).delete_endpoint(delete_endpoint_config=True)
model.delete_model()
print("deleted endpoint", endpoint_name, "(and its config) and model", model.name)

# boto3.resource("s3").Bucket(bucket).objects.filter(Prefix=prefix).delete()